In [1]:
# replication setup — added by patch_notebook.py
import os, sys
_HERE = os.path.abspath(os.getcwd())
for p in (os.path.abspath(os.path.join(_HERE, "..", "..", "reference")),
          os.path.abspath(os.path.join(_HERE, ".."))):
    if p not in sys.path:
        sys.path.insert(0, p)

# networkx 3.x removed from_numpy_matrix; reference/stats_count.py still calls it.
# Behavior of from_numpy_array on a 2D ndarray matches the old from_numpy_matrix.
import networkx as _nx
if not hasattr(_nx, "from_numpy_matrix"):
    _nx.from_numpy_matrix = _nx.from_numpy_array

# numpy 1.20 deprecated np.int as an alias for builtin int; 1.24 removed it.
# reference/ripser_count.py still calls .astype(np.int). Restore the alias.
import numpy as _np
if not hasattr(_np, "int"):
    _np.int = int


<!-- replication disk-budget — added by patch_notebook.py -->
## Disk budget

This notebook persists attention tensors to `../outputs/attentions/` before
extracting topology features. Each subset writes ~4.7GB
(samples × 12 layers × 12 heads × 128² × float16). Running all three splits
(train, valid, test) without cleanup needs ~14GB free.

The final cell deletes the per-subset attention files after features are
saved. Set `KEEP_ATTENTION = True` in that cell to retain them — useful
when chaining into `features_calculation_ripser_and_templates.ipynb` on
the same subset to avoid recomputing attention.


In [2]:
from collections import defaultdict
import itertools
import re
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from transformers import BertTokenizer, BertModel, BertForSequenceClassification

from stats_count import *
from grab_weights_compat import grab_attention_weights, text_preprocessing

In [3]:
import warnings

warnings.filterwarnings('ignore')

In [4]:
!env | grep CUDA_VISIBLE

## Parameters

In [5]:
np.random.seed(42) # For reproducibility.

In [6]:
max_tokens_amount  = 128 # The number of tokens to which the tokenized text is truncated / padded.
stats_cap          = 500 # Max value that the feature can take. Is NOT applicable to Betty numbers.
    
layers_of_interest = [i for i in range(12)]  # Layers for which attention matrices and features on them are 
                                             # calculated. For calculating features on all layers, leave it be
                                             # [i for i in range(12)].
stats_name = "s_e_v_c_b0b1" # The set of topological features that will be count (see explanation below)

thresholds_array = [0.025, 0.05, 0.1, 0.25, 0.5, 0.75] # The set of thresholds
thrs = len(thresholds_array)                           # ("t" in the paper)

model_path = tokenizer_path = "bert-base-uncased"  

# You can use either standard or fine-tuned BERT. If you want to use fine-tuned BERT to your current task, save the
# model and the tokenizer with the commands tokenizer.save_pretrained(output_dir); 
# bert_classifier.save_pretrained(output_dir) into the same directory and insert the path to it here.

### Explanation of stats_name parameter

Currently, we implemented calculation of the following graphs features:
* "s"    - amount of strongly connected components
* "w"    - amount of weakly connected components
* "e"    - amount of edges
* "v"    - average vertex degree
* "c"    - amount of (directed) simple cycles
* "b0b1" - Betti numbers

The variable stats_name contains a string with the names of the features, which you want to calculate. The format of the string is the following:

"stat_name + "_" + stat_name + "_" + stat_name + ..."

**For example**:

`stats_name == "s_w"` means that the number of strongly and weakly connected components will be calculated

`stats_name == "b0b1"` means that only the Betti numbers will be calculated

`stats_name == "b0b1_c"` means that Betti numbers and the number of simple cycles will be calculated

e.t.c.

## Filenames

In [7]:
subset = "train"           # .csv file with the texts, for which we count topological features
input_dir = "../data/processed/"  # Name of the directory with .csv file
output_dir = "../outputs/" # Name of the directory with calculations results

prefix = output_dir + subset

r_file     = output_dir + 'attentions/' + subset  + "_all_heads_" + str(len(layers_of_interest)) + "_layers_MAX_LEN_" + \
             str(max_tokens_amount) + "_" + model_path.split("/")[-1]
# Name of the file for attention matrices weights

stats_file = output_dir + 'features/' + subset + "_all_heads_" + str(len(layers_of_interest)) + "_layers_" + stats_name \
             + "_lists_array_" + str(thrs) + "_thrs_MAX_LEN_" + str(max_tokens_amount) + \
             "_" + model_path.split("/")[-1] + '.npy'
# Name of the file for topological features array

In [8]:
stats_file

'../outputs/features/train_all_heads_12_layers_s_e_v_c_b0b1_lists_array_6_thrs_MAX_LEN_128_bert-base-uncased.npy'

In [9]:
r_file

'../outputs/attentions/train_all_heads_12_layers_MAX_LEN_128_bert-base-uncased'

.csv file must contain the column with the name **sentence** with the texts. It can also contain the column **labels**, which will be needed for testing. Any other arbitrary columns will be ignored.

In [10]:
try:
    data = pd.read_csv(input_dir + subset + ".csv").reset_index(drop=True)
except:
    #data = pd.read_csv(input_dir + subset + ".tsv", delimiter="\t")
    data = pd.read_csv(input_dir + subset + ".tsv", delimiter="\t", header=None)
    data.columns = ["0", "labels", "2", "sentence"]

In [11]:
data.head()

,sentence,labels
0,TRENTON -- NJ Transit spokesman Alan Brock tol...,1
1,Merkley Leads Over 200 Members of Congress in ...,0
2,Cypher and Vicky actually believe that the LOK...,1
3,"In addition to Orlando, Planet explore Upward ...",1
4,"Statewide elections, gubernatorial elections e...",0


In [12]:
sentences = data['sentence']
print("Average amount of words in example:", \
      np.mean(list(map(len, map(lambda x: re.sub('\w', ' ', x).split(" "), data['sentence'])))))
print("Max. amount of words in example:", \
      np.max(list(map(len, map(lambda x: re.sub('\w', ' ', x).split(" "), data['sentence'])))))
print("Min. amount of words in example:", \
      np.min(list(map(len, map(lambda x: re.sub('\w', ' ', x).split(" "), data['sentence'])))))

Average amount of words in example: 2594.103
Max. amount of words in example: 5465
Min. amount of words in example: 1


In [13]:
def get_token_length(batch_texts):
    inputs = tokenizer(list(batch_texts),
       return_tensors='pt',
       add_special_tokens=True,
       max_length=MAX_LEN,             # Max length to truncate/pad
       padding="max_length",         # Pad sentence to max length
       truncation=True
    )
    inputs = inputs['input_ids'].numpy()
    n_tokens = []
    indexes = np.argwhere(inputs == tokenizer.pad_token_id)
    for i in range(inputs.shape[0]):
        ids = indexes[(indexes == i)[:, 0]]
        if not len(ids):
            n_tokens.append(MAX_LEN)
        else:
            n_tokens.append(ids[0, 1])
    return n_tokens

In [14]:
MAX_LEN = max_tokens_amount
tokenizer = BertTokenizer.from_pretrained(tokenizer_path, do_lower_case=True)

In [15]:
data['tokenizer_length'] = get_token_length(data['sentence'].values)

In [16]:
data

,sentence,labels,tokenizer_length
0,TRENTON -- NJ Transit spokesman Alan Brock tol...,1,128
1,Merkley Leads Over 200 Members of Congress in ...,0,128
2,Cypher and Vicky actually believe that the LOK...,1,128
3,"In addition to Orlando, Planet explore Upward ...",1,128
4,"Statewide elections, gubernatorial elections e...",0,128
...,...,...,...
995,"Question as Bob Corker makes ""Trump BS"" commen...",1,128
996,Tonight the Bulls took on a Pistons team that ...,0,128
997,Lt. Phil Robertson began discussing the topic ...,1,128
998,Thread Options\n\njasper34 Misery PR Lead Anch...,0,128


In [17]:
ntokens_array = data['tokenizer_length'].values

## Attention extraction

Loading **BERT** and tokenizers using **transformers** library.

In [18]:
from math import ceil

batch_size = 10 # batch size
number_of_batches = ceil(len(data['sentence']) / batch_size)
DUMP_SIZE = 100 # number of batches to be dumped
batched_sentences = np.array_split(data['sentence'].values, number_of_batches)
number_of_files = ceil(number_of_batches / DUMP_SIZE)
adj_matricies = []
adj_filenames = []
assert number_of_batches == len(batched_sentences) # sanity check

In [19]:
device='cuda:0'
model = BertForSequenceClassification.from_pretrained(model_path, output_attentions=True)
tokenizer = BertTokenizer.from_pretrained(tokenizer_path, do_lower_case=True)
model = model.to(device)
MAX_LEN = max_tokens_amount

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
for i in tqdm(range(number_of_batches), desc="Weights calc"): 
    attention_w = grab_attention_weights(model, tokenizer, batched_sentences[i], max_tokens_amount, device)
    # sample X layer X head X n_token X n_token
    adj_matricies.append(attention_w)
    if (i+1) % DUMP_SIZE == 0: # dumping
        print(f'Saving: shape {adj_matricies[0].shape}')
        adj_matricies = np.concatenate(adj_matricies, axis=1)
        print("Concatenated")
        adj_matricies = np.swapaxes(adj_matricies, axis1=0, axis2=1) # sample X layer X head X n_token X n_token
        filename = r_file + "_part" + str(ceil(i/DUMP_SIZE)) + "of" + str(number_of_files) + '.npy'
        print(f"Saving weights to : {filename}")
        adj_filenames.append(filename)
        np.save(filename, adj_matricies)
        adj_matricies = []
        
if len(adj_matricies):
    filename = r_file + "_part" + str(ceil(i/DUMP_SIZE)) + "of" + str(number_of_files) + '.npy'
    print(f'Saving: shape {adj_matricies[0].shape}')
    adj_matricies = np.concatenate(adj_matricies, axis=1)
    print("Concatenated")
    adj_matricies = np.swapaxes(adj_matricies, axis1=0, axis2=1) # sample X layer X head X n_token X n_token
    print(f"Saving weights to : {filename}")
    np.save(filename, adj_matricies)

print("Results saved.")

Weights calc:   0%|          | 0/100 [00:00<?, ?it/s]

Saving: shape (12, 10, 12, 128, 128)
Concatenated
Saving weights to : ../outputs/attentions/train_all_heads_12_layers_MAX_LEN_128_bert-base-uncased_part1of1.npy
Results saved.


## Calculating topological features

In [21]:
stats_name.split("_")

['s', 'e', 'v', 'c', 'b0b1']

In [22]:
import os
from multiprocessing import Pool
from tqdm import tqdm

adj_filenames = [
    output_dir + 'attentions/' + filename 
    for filename in os.listdir(output_dir + 'attentions/') if r_file in (output_dir + 'attentions/' + filename)
]
# sorted by part number
adj_filenames = sorted(adj_filenames, key = lambda x: int(x.split('_')[-1].split('of')[0][4:].strip())) 
adj_filenames

['../outputs/attentions/train_all_heads_12_layers_MAX_LEN_128_bert-base-uncased_part1of1.npy']

In [23]:
# What is calculated in "f(v)". You can add any other function from the array with vertex degrees.

def function_for_v(list_of_v_degrees_of_graph):
    return sum(map(lambda x: np.sqrt(x*x), list_of_v_degrees_of_graph))

def split_matricies_and_lengths(adj_matricies, ntokens_array, num_of_workers):
    splitted_adj_matricies = np.array_split(adj_matricies, num_of_workers)
    splitted_ntokens = np.array_split(ntokens_array, num_of_workers)
    assert all([len(m)==len(n) for m, n in zip(splitted_adj_matricies, splitted_ntokens)]), "Split is not valid!"
    return zip(splitted_adj_matricies, splitted_ntokens)

In [24]:
import gc
try:
    del model  # not needed for topology features (read from disk)
except NameError:
    pass
try:
    import torch
    torch.cuda.empty_cache()
except Exception:
    pass
gc.collect()

num_of_workers = 8
pool = Pool(num_of_workers)

In [25]:
stats_tuple_lists_array = []
for i, filename in enumerate(tqdm(adj_filenames, desc='Вычисление признаков')):
    adj_matricies = np.load(filename, allow_pickle=True)
    ntokens = ntokens_array[i*batch_size*DUMP_SIZE : (i+1)*batch_size*DUMP_SIZE]
    splitted = split_matricies_and_lengths(adj_matricies, ntokens, num_of_workers)
    args = [(m, thresholds_array, ntokens, stats_name.split("_"), stats_cap) for m, ntokens in splitted]
    stats_tuple_lists_array_part = pool.starmap(
        count_top_stats, args
    )
    stats_tuple_lists_array.append(np.concatenate([_ for _ in stats_tuple_lists_array_part], axis=3))

Вычисление признаков: 100%|██████████| 1/1 [07:15<00:00, 435.57s/it]


In [26]:
stats_tuple_lists_array = np.concatenate(stats_tuple_lists_array, axis=3)

In [27]:
stats_tuple_lists_array.shape

(12, 12, 6, 1000, 6)

In [28]:
from numpy import inf

np.sum(stats_tuple_lists_array[stats_tuple_lists_array == -inf]) + \
np.sum(stats_tuple_lists_array[stats_tuple_lists_array == inf])

0.0

In [29]:
stats_file

'../outputs/features/train_all_heads_12_layers_s_e_v_c_b0b1_lists_array_6_thrs_MAX_LEN_128_bert-base-uncased.npy'

In [30]:
np.save(stats_file, stats_tuple_lists_array)

##### Checking the size of features matrices:

Layers amount **Х** Heads amount **Х** Features amount **X** Examples amount **Х** Thresholds amount

**For example**:

`stats_name == "s_w"` => `Features amount == 2`

`stats_name == "b0b1"` => `Features amount == 2`

`stats_name == "b0b1_c"` => `Features amount == 3`

e.t.c.

`thresholds_array == [0.025, 0.05, 0.1, 0.25, 0.5, 0.75]` => `Thresholds amount == 6`

In [31]:
stats_tuple_lists_array.shape

(12, 12, 6, 1000, 6)

In [ ]:
# replication cleanup — added by patch_notebook.py
# Delete per-subset attention .npy files now that features are saved.
# Set KEEP_ATTENTION = True to retain them (e.g. to skip recompute when
# running features_calculation_ripser_and_templates.ipynb on the same
# subset). See the disk-budget cell at the top.
KEEP_ATTENTION = False

import os as _os
if KEEP_ATTENTION:
    print(f"Cleanup: KEEP_ATTENTION=True, retaining {len(adj_filenames)} attention file(s) for subset={subset!r}.")
else:
    _removed = 0
    for _f in adj_filenames:
        if _os.path.exists(_f):
            _os.remove(_f)
            _removed += 1
    print(f"Cleanup: removed {_removed} attention file(s) for subset={subset!r}.")
